# Final full-training refit: LightGBM + NLP

Run All fits **only** the LightGBM+NLP candidate selected by the completed 10-fold CV in `notebooks/02_model_comparison_cv.ipynb`. It loads the complete merged Parquet file once with `pd.read_parquet()`, then selects 2010–2021 rows with an exact-binary target, including edited reviews. It uses the same 96 predictors, language preprocessing, and median of the ten recorded `best_iteration` values. The exported Booster text file and preprocessing metadata are written under `runs/training/`.

The 10-fold CV excluded edited reviews, so its metrics do not validate this broader final-training cohort. Edited review text may include changes made after the outcome. This Notebook does not calculate new cross-validation metrics or select a new model. It loads the source into RAM; ensure enough memory is available before Run All. The `.txt` file expects the transformation described in its adjacent `.json` file at inference time. The submission copy locates the project relative to the working directory; `TEAM18_PROJECT_ROOT` can override it. New exports go to `runs/training/`. Run with a Python 3.12 kernel using the supplied project requirements.

## 1. Paths and environment

In [ ]:
import os
import sys
import time
import gc
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path((os.environ.get("TEAM18_PROJECT_ROOT") or next(
    str(p) for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "run.py").is_file() and (p / "src" / "steam_reviews").is_dir()
))).resolve()
TRAINING_DIR = PROJECT_ROOT / "src"
if str(TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR))

from steam_reviews.modeling.final_training import (
    load_approved_contract,
    prepare_training_frame,
    train_frame_and_export,
)

OUTPUT_DIR = Path(os.environ.get("TRAINING_OUTPUT_DIR", PROJECT_ROOT / "runs" / "training"))
MODEL_PATH = OUTPUT_DIR / "training_full_model.txt"
METADATA_PATH = OUTPUT_DIR / "training_full_model.json"
THREADS = int(os.environ.get("TRAINING_THREADS", "4"))
started = time.perf_counter()
print("Output model:", MODEL_PATH)

## 2. Verify the completed CV and load the merged source once

In [ ]:
contract = load_approved_contract(PROJECT_ROOT)
print("CV reference rows (unedited only):", f"{contract['row_count']:,}")
print("CV reference positive rows:", f"{contract['positive_rows']:,}")
print("Selected rounds:", contract["final_rounds"])
print("10-fold best iterations:", contract["fold_iterations"])
print("Source SHA-256:", contract["source_sha256"])
df_merged = pd.read_parquet(contract["source_path"])
train_start = pd.Timestamp("2010-10-15", tz="UTC")
val_start = pd.Timestamp("2022-01-01", tz="UTC")
# test_start = pd.Timestamp("2023-01-01", tz="UTC")
review_dates = pd.to_datetime(df_merged["review_created_utc"], utc=True, errors="raise")
print("Raw training-window rows:", f"{((review_dates >= train_start) & (review_dates < val_start)).sum():,}")

training = prepare_training_frame(df_merged, contract["input_features"])
del df_merged, review_dates
gc.collect()
print("Eligible training rows:", f"{len(training):,}")
print("Eligible positive rows:", f"{int(training['still_playing_d30'].sum()):,}")
print("Additional rows beyond CV cohort:", f"{len(training) - contract['row_count']:,}")

## 3. Fit on every eligible training row and export the Booster

The completed 10-fold run supplies the model configuration, but the final cohort now includes edited reviews and therefore has different counts. This cell fits directly from the in-memory training frame using the approved language imputer, ordinal encoder, and LightGBM settings. It writes the `.txt` model atomically, then saves feature order, language mapping, and cohort definition beside it.

In [ ]:
metadata = train_frame_and_export(
    training=training,
    model_path=MODEL_PATH,
    metadata_path=METADATA_PATH,
    input_features=contract["input_features"],
    rounds=contract["final_rounds"],
    threads=THREADS,
    source_sha256=contract["source_sha256"],
    staging_signature=contract["staging_signature"],
)
print("Exported:", MODEL_PATH)
print("Preprocessing metadata:", METADATA_PATH)
print("Trained rows:", f"{metadata['rows']:,}")
print("Trees:", metadata["rounds_trained"])
print("Model SHA-256:", metadata["model_txt_sha256"])
print("Elapsed seconds:", round(time.perf_counter() - started, 1))